# w30_cpu1_multidur_and_imptc_pcr6
Inputs: `shortpose-wave1-private-inputs` | Accelerator: `None` (CPU) | Internet: `On`


In [ ]:
import hashlib, os, subprocess, sys, tempfile, zipfile
from pathlib import Path

WORK, MOUNT = Path('/kaggle/working'), Path('/kaggle/input')
assert WORK.is_dir(), 'Kaggle Cloud environment required'
REV = '60652e2f873c05c435de04d10f0e819a4a3d122f'
with tempfile.TemporaryDirectory(prefix='w30_c1_') as ts:
    tmp = Path(ts)
    repo, meta_dir = tmp / 'repo', tmp / 'pinns_meta'
    subprocess.run(['git', 'init', '-q', str(repo)], check=True)
    subprocess.run(['git', '-C', str(repo), 'fetch', '-q', '--depth=1', 'https://github.com/pjyyyds/Unstructured-ShortPose-Pred.git', REV], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '-q', '--detach', 'FETCH_HEAD'], check=True)
    env4 = {**os.environ, 'PYTHONPATH': str(repo), 'PYTHONDONTWRITEBYTECODE': '1', 'OMP_NUM_THREADS': '4', 'MKL_NUM_THREADS': '4'}
    joint_bin = next(MOUNT.rglob('joint_actions_all.bin'))
    heldout_bin = next(MOUNT.rglob('shortpose_heldout.bin'))
    map_bin = next(MOUNT.rglob('pinns_source_label_causal_maps.bin'))
    matched_npz = next(MOUNT.rglob('pinns_matched_histories.npz'))
    pose_npz = tmp / 'pose.npz'
    with zipfile.ZipFile(heldout_bin) as zf:
        pose_npz.write_bytes(zf.read('pose.npz'))
    obs_hits = list(MOUNT.rglob('matched_observations.npz'))
    if obs_hits:
        matched_obs = obs_hits[0]
    else:
        md_zips = list(MOUNT.rglob('matched_durations.zip'))
        matched_obs = tmp / 'matched_observations.npz'
        with zipfile.ZipFile(md_zips[0]) as zf:
            nm = next(n for n in zf.namelist() if n.endswith('matched_observations.npz'))
            matched_obs.write_bytes(zf.read(nm))
    subprocess.run([sys.executable, '-m', 'scripts.stage_pinns_metadata', '--output-dir', str(meta_dir)], cwd=repo, env=env4, check=True)
    imptc_tar = tmp / 'imptc_samples.tar.gz'
    out_zip = WORK / 'wave30_multidur_and_imptc_pcr6.zip'
    subprocess.run([
        sys.executable, '-u', '-m', 'scripts.run_wave30_multidur_and_imptc_pcr6',
        '--matched-npz', str(matched_npz), '--matched-obs-npz', str(matched_obs),
        '--meta-dir', str(meta_dir), '--pose-npz', str(pose_npz),
        '--joint-zip', str(joint_bin), '--causal-map-bin', str(map_bin),
        '--imptc-tar', str(imptc_tar), '--zip-path', str(out_zip)
    ], cwd=repo, env=env4, check=True)
    print('DONE:', out_zip, out_zip.stat().st_size, hashlib.sha256(out_zip.read_bytes()).hexdigest(), flush=True)
